# ⚡ VatioPlan — Programar la producción según el precio de la energía
**Herramientas de Inteligencia Artificial · Universidad de los Andes · Proyecto final 2026**

Este notebook recorre **todas las fases del proyecto**, en el mismo orden de la rúbrica:

| # | Fase | Herramienta |
|---|------|-------------|
| 1 | Problema | — |
| 2 | Información de entrada (descarga automática) | API SIMEM/XM, NOAA |
| 3 | Valor de los datos (dato → resultado → *insight*) | pandas |
| 4 | Machine Learning: pronóstico del precio | scikit-learn |
| 5 | Reglas de ingeniería + optimización | PuLP (programación entera) |
| 6 | Valor de la propuesta: ahorro con precios reales (*backtest*) | — |
| 7 | IA generativa: informe para el jefe de planta | Gemini / Groq (gratis) |
| 8 | Orquestación y tablero | Prefect / n8n + Streamlit |

> **Todo es gratuito.** Funciona en Google Colab o en Jupyter local.

## 0. Preparación

In [ ]:
# Si estás en Google Colab: sube el archivo vatioplan.zip cuando lo pida esta celda.
import os, sys
try:
    import google.colab  # noqa
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and not os.path.exists("vatioplan"):
    from google.colab import files
    print("Sube vatioplan.zip ...")
    files.upload()
    !unzip -q -o vatioplan.zip
if EN_COLAB:
    %cd vatioplan
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")   # trabajar desde la raíz del proyecto
sys.path.insert(0, os.getcwd())
print("Carpeta de trabajo:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (12, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})

from vatioplan.datos import cargar_datos
from vatioplan.modelo import evaluar, entrenar, pronosticar, importancia_variables
from vatioplan.planta import (Planta, plan_actual, optimizar_dia, validar_plan, costo_plan,
                              perfil_potencia, planear_periodo, backtest)
from vatioplan.pipeline import resumen_para_ia, paso_pronostico
from vatioplan.informe import generar_informe

## 1. El problema

En Colombia la energía se compra en el **mercado mayorista**, donde el **precio de bolsa cambia cada hora**.
Cerca del 70 % de la generación es hidráulica: el precio sube con la sequía (El Niño) y todos los días
tiene un **pico nocturno (18–21 h)**, cuando la demanda es máxima.

Las plantas industriales programan sus equipos **sin mirar el precio**. Muchas cargas se podrían mover
de horario sin afectar la producción: hornos de secado, molinos, chillers con almacenamiento de frío,
bombeo a tanques, carga de montacargas.

**Pregunta:** *¿a qué hora debería funcionar cada equipo para producir lo mismo pagando menos energía?*

**Usuario:** jefe de planta / programador de producción · **Cliente:** empresas industriales cuyo costo
de energía depende del precio de bolsa (usuarios no regulados o con contratos indexados).

## 2. Información de entrada — descarga automática

| Fuente | Qué aporta | Acceso |
|---|---|---|
| **SIMEM (XM)**, conjunto `EC6945` | Precio de bolsa nacional **horario** | API pública GET, sin clave |
| **API XM** `PorcVoluUtilDiar`, `AporEner` | Nivel de embalses y aportes hídricos (diario) | API pública POST, sin clave |
| **NOAA CPC** | Índice ONI (El Niño / La Niña), mensual | Archivo de texto público |
| Festivos de Colombia | Calendario | Librería `holidays` |

La función `cargar_datos` descarga todo, se queda con la **versión más reciente** de cada precio
(XM revisa los precios varias veces) y guarda una **caché** en `data/` para solo bajar lo nuevo.

> Si no hay internet, genera datos **SIMULADOS** para poder probar el código (quedan marcados).

In [ ]:
df = cargar_datos(inicio="2023-01-01")
FUENTE = df.attrs["fuente"]
print("\nFuente de datos:", FUENTE)
df.tail()

### Levantamiento de los datos (tabla del anteproyecto, calculada con los datos reales)

In [ ]:
ultimo = df["precio"].last_valid_index()
tabla = pd.DataFrame([
    {"Fuente": "Precio de bolsa (SIMEM)", "Existe": "✓", "Acceso": "✓ API pública",
     "Registros": f"{df['precio'].notna().sum():,} horas",
     "Faltantes": f"{df['precio'].isna().mean():.2%}",
     "Desde–hasta": f"{df.index.min():%Y-%m-%d} → {ultimo:%Y-%m-%d}", "Automatizable": "✓"},
] + [
    {"Fuente": nombre, "Existe": "✓" if col in df else "✗", "Acceso": "✓" if col in df else "✗",
     "Registros": f"{df[col].notna().sum() // 24:,} días" if col in df else "—",
     "Faltantes": f"{df[col].isna().mean():.2%}" if col in df else "—",
     "Desde–hasta": "diario" if col in df else "—", "Automatizable": "✓" if col in df else "✗"}
    for nombre, col in [("Embalses % (XM)", "vol_util_pct"), ("Aportes hídricos (XM)", "aportes_gwh"),
                        ("ONI El Niño (NOAA)", "oni")]
])
print(f"Retraso de publicación: el último precio es de hace {(pd.Timestamp.now() - ultimo).days} días")
tabla

## 3. Valor de los datos: dato → resultado → *insight*
El objetivo no es solo procesar datos sino encontrar patrones que ayuden a **decidir**.

In [ ]:
p = df["precio"].dropna()
ult = p[p.index >= p.index.max() - pd.Timedelta(days=365)]    # último año

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
perfil = ult.groupby([ult.index.hour]).agg(["mean", lambda s: s.quantile(.1), lambda s: s.quantile(.9)])
perfil.columns = ["media", "p10", "p90"]
ax[0].fill_between(perfil.index, perfil.p10, perfil.p90, alpha=.25, color="#d9480f", label="10–90 %")
ax[0].plot(perfil.index, perfil.media, color="#d9480f", lw=2.5, label="Promedio")
ax[0].set(title="Precio por hora del día (último año)", xlabel="Hora", ylabel="COP/kWh", xticks=range(0, 24, 2))
ax[0].legend()
nombres = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
sem = ult.groupby(ult.index.dayofweek).mean()
ax[1].bar(nombres, sem.values, color=["#868e96"] * 6 + ["#2f9e44"])
ax[1].set(title="Precio promedio por día de la semana", ylabel="COP/kWh")
plt.tight_layout(); plt.show()

diario = ult.resample("D").agg(["min", "max"])
ratio = (diario["max"] / diario["min"]).median()
barata, cara = perfil.media.idxmin(), perfil.media.idxmax()
print(f"DATO: en un día típico la hora más cara cuesta {ratio:.1f} veces la más barata.")
print(f"RESULTADO: la hora más cara en promedio es las {cara}:00 y la más barata las {barata}:00 "
      f"({perfil.media[cara] / perfil.media[barata] - 1:.0%} más cara).")
print("INSIGHT: mover consumo flexible fuera de las 18–21 h reduce costos SIN producir menos.")

In [ ]:
# Hidrología y clima: ¿por qué hay semanas enteras caras?
if "vol_util_pct" in df:
    d = df.resample("W").mean(numeric_only=True).dropna(subset=["precio"])
    fig, ax1 = plt.subplots(figsize=(14, 4))
    ax1.plot(d.index, d["precio"], color="#d9480f", lw=2, label="Precio semanal")
    ax1.set_ylabel("COP/kWh", color="#d9480f")
    ax2 = ax1.twinx()
    ax2.plot(d.index, d["vol_util_pct"], color="#1971c2", lw=2, label="Embalses (%)")
    ax2.set_ylabel("Volumen útil embalses (%)", color="#1971c2"); ax2.grid(False)
    plt.title("Cuando los embalses bajan, el precio sube"); plt.show()
    print(f"Correlación semanal precio–embalses: {d['precio'].corr(d['vol_util_pct']):.2f}")
    print("INSIGHT: la hidrología anticipa semanas caras → conviene adelantar producción de inventario.")

## 4. Machine Learning: pronóstico del precio
**Reto:** el precio se publica con ~3 días de retraso y queremos planear varios días adelante.
Por eso el modelo solo usa información de **hace 7 días o más** (sin fuga de datos).

**Cuidado con las correlaciones engañosas:** la demanda o la generación del mismo momento se
correlacionan con el precio, pero se deciden en el mismo despacho; usarlas sería "hacer trampa".

- **Modelo:** `HistGradientBoostingRegressor` (árboles de gradiente, scikit-learn).
- **Línea base:** "ingenuo" = el precio de la misma hora hace 7 días. Si no le ganamos, el ML no vale la pena.
- **Prueba:** entrenar con todo menos los **últimos 90 días** y evaluar en ellos.

In [ ]:
ev = evaluar(df, dias_prueba=90)
m = ev["metricas"]
pd.DataFrame({
    "Modelo ML": [m["mae_modelo"], m["mape_modelo"], m["acierto_horas_baratas_modelo"] * 100],
    "Ingenuo (hace 7 días)": [m["mae_base"], m["mape_base"], m["acierto_horas_baratas_base"] * 100],
}, index=["MAE (COP/kWh)", "MAPE (%)", "Horas baratas acertadas (%)"]).round(1)

In [ ]:
v = pd.DataFrame({"Real": ev["real"], "Modelo ML": ev["pred"], "Ingenuo": ev["base"]}).tail(24 * 14)
ax = v.plot(color=["#212529", "#d9480f", "#adb5bd"], lw=1.6, figsize=(14, 4))
ax.set(title="Últimas 2 semanas del periodo de prueba", ylabel="COP/kWh")
plt.show()
print(f"El modelo reduce el error {m['mejora_mae_pct']:.0f}% frente al método ingenuo.")
print(f"Identifica {m['acierto_horas_baratas_modelo']:.0%} de las 8 horas más baratas de cada día "
      f"(ingenuo: {m['acierto_horas_baratas_base']:.0%}). Esto es lo que importa para programar la planta.")

In [ ]:
imp = importancia_variables(ev["modelo"], df)
imp.head(10)[::-1].plot.barh(color="#1971c2", figsize=(8, 4), title="¿Qué variables usa el modelo?")
plt.xlabel("Aumento del error si se desordena la variable (COP/kWh)"); plt.show()

### Pronóstico de los próximos días (modelo re-entrenado con todos los datos)

In [ ]:
modelo = entrenar(df)
pron = paso_pronostico(df, modelo, dias_plan=3)
hist = df["precio"].tail(24 * 10)
plt.plot(hist.index, hist.values, color="#212529", label="Real (último publicado)")
plt.plot(pron.index, pron.values, color="#d9480f", ls="--", lw=2, label="Pronóstico (días a planear)")
plt.title("Precio de bolsa: historia reciente y pronóstico"); plt.ylabel("COP/kWh"); plt.legend(); plt.show()

## 5. Reglas de ingeniería + optimización
Planta de ejemplo (editable en `config/planta_ejemplo.json`): inyección de plásticos, 2 turnos, 700 kW máximos.

Se resuelve un **problema de programación entera** (PuLP, gratis) que minimiza el costo sujeto a:

| Regla | Significado |
|---|---|
| R1 | Cada equipo cumple sus horas diarias (no se sacrifica producción) |
| R2 | Solo opera en su ventana permitida (turnos, operarios) |
| R3 | El horno opera en un bloque continuo |
| R4 | La potencia total no supera la potencia contratada |

In [ ]:
planta = Planta.desde_json("config/planta_ejemplo.json")
pd.DataFrame([{"Equipo": c.nombre, "kW": c.potencia_kw, "Horas/día": c.horas_dia,
               "Continua": "Sí" if c.continua else "No", "Horas permitidas": c.horas_permitidas,
               "Motivo": c.motivo_restriccion} for c in planta.cargas])

In [ ]:
dia = sorted(set(pron.index.date))[0]
precios = pron[pron.index.date == dia].values
act, opt = plan_actual(planta), optimizar_dia(precios, planta)

colores = ["#ced4da", "#d9480f", "#1971c2", "#2f9e44", "#7048e8", "#e67700"]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (titulo, plan) in zip(axes, [("Operación actual", act), ("VatioPlan (optimizado)", opt)]):
    perf = perfil_potencia(plan, planta)
    perf.plot.bar(stacked=True, ax=ax, color=colores, width=.9, legend=False)
    ax.axhline(planta.potencia_maxima_kw, color="k", ls="--", lw=1)
    ax.set_ylim(0, planta.potencia_maxima_kw * 1.15)
    ax2 = ax.twinx(); ax2.plot(range(24), precios, color="k", lw=2); ax2.grid(False)
    ax2.set_ylabel("COP/kWh")
    ax.set(title=f"{titulo} · costo ${costo_plan(plan, precios, planta):,.0f}", xlabel="Hora", ylabel="kW")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()
print(f"Ahorro del día {dia}: ${costo_plan(act, precios, planta) - costo_plan(opt, precios, planta):,.0f} COP")

In [ ]:
validar_plan(opt, planta)[["regla", "carga", "resultado"]]

## 6. Valor de la propuesta: ¿cuánto se habría ahorrado de verdad? (*backtest*)
Durante los 90 días de prueba, cada día se planeó **con el pronóstico** y el costo se calculó **con el precio real**.
Se compara contra la operación actual y contra el "plan perfecto" (si se conociera el precio de antemano).

In [ ]:
bt = backtest(ev["real"], ev["pred"], planta)
tot = bt.sum()
ahorro_pron = 1 - tot.costo_con_pronostico / tot.costo_actual
ahorro_max = 1 - tot.costo_perfecto / tot.costo_actual
ax = (bt["costo_actual"] - bt["costo_con_pronostico"]).cumsum().div(1e6).plot(
    color="#2f9e44", lw=2.5, figsize=(12, 4), label="Con pronóstico (real)")
(bt["costo_actual"] - bt["costo_perfecto"]).cumsum().div(1e6).plot(ax=ax, color="#adb5bd", ls="--", label="Máximo teórico")
ax.set(title="Ahorro acumulado en cargas flexibles", ylabel="Millones de COP"); ax.legend(); plt.show()

anual = (tot.costo_actual - tot.costo_con_pronostico) / len(bt) * 365
print(f"Ahorro real con pronóstico: {ahorro_pron:.1%} del costo de las cargas flexibles")
print(f"Máximo teórico (precio perfecto): {ahorro_max:.1%} → el modelo captura {ahorro_pron / ahorro_max:.0%} del potencial")
print(f"Equivale a ≈ ${anual:,.0f} COP al año para esta planta de ejemplo (con exposición a bolsa = {planta.exposicion_bolsa:.0%})")

## 7. IA generativa: informe para el jefe de planta
El LLM recibe **solo los números calculados** (JSON) y se le prohíbe inventar cifras: el modelo y las
reglas deciden; la IA **redacta y prioriza**.

Opcional — API key gratuita:
- Gemini: https://aistudio.google.com/apikey → `GEMINI_API_KEY`
- Groq: https://console.groq.com/keys → `GROQ_API_KEY`

Sin key se usa una plantilla (el pipeline nunca se detiene).

In [ ]:
# os.environ["GEMINI_API_KEY"] = "pega-aqui-tu-key"     # ← opcional
plan = planear_periodo(pron, planta)
resumen = resumen_para_ia(plan, pron, ev, planta, FUENTE)
print(json.dumps(resumen, ensure_ascii=False, indent=1)[:1200], "...")

In [ ]:
from IPython.display import Markdown, display
texto, proveedor = generar_informe(resumen)
print("Generado con:", proveedor)
display(Markdown(texto))

## 8. Orquestación y tablero

**Orquestación (todos los días 6:00 a. m.)**, después de que XM actualiza (~4:00 a. m.):

```
Descargar datos → Entrenar/pronosticar → Optimizar + reglas → Informe IA → Publicar/Notificar
```
- **Prefect** (`flujo_prefect.py`): cada paso es una tarea con reintentos; si el plan viola una regla, no se publica.
  `python flujo_prefect.py --una-vez` para probar · `python flujo_prefect.py` para dejarlo programado.
- **n8n** (`n8n_workflow.json`): versión visual; se importa en n8n y envía el informe por correo.

**Tablero low-code (Streamlit):** `streamlit run app.py` — plan del día, pronóstico, validación y el informe IA.
Se publica gratis en https://share.streamlit.io conectando el repositorio de GitHub.

In [ ]:
# Ejecutar el pipeline completo, tal como lo haría el orquestador:
from vatioplan.pipeline import ejecutar
salida = ejecutar(verbose=False)
print("Archivos generados:", sorted(os.listdir("salidas")))

## 9. Conclusiones (se llenan con los resultados de arriba)
1. **Problema:** el precio de la energía varía varias veces dentro del día y la industria no lo aprovecha.
2. **Datos:** públicos, horarios, gratuitos y automatizables (SIMEM/XM + NOAA).
3. **ML:** el modelo supera al método ingenuo y, sobre todo, identifica bien las horas baratas.
4. **Reglas:** el plan siempre cumple turnos, continuidad y potencia máxima (validado automáticamente).
5. **Valor:** ahorro medido con precios reales en el backtest, sin producir menos.
6. **Innovación:** tecnología existente (pronóstico + optimización + LLM) aplicada a la programación de producción.